# Notebook 05 — Fill in Your Custom Agent Harness

You've evaluated two off-the-shelf coding agents (Claude Code and Kiro) in notebooks 06 and 07. Now you build your **own**, scored on the same eval set.

The point isn't to invent the wheel. We ship a **harness skeleton** at `my_agent/` that handles all the eval-side plumbing:
- CLI argument parsing (`--task-id`, `--tasks-file`, `--repo`, `--out`, `--trace-out`, `--seed`)
- Loading the task from the YAML
- Capturing the diff after the agent runs
- Serializing the tool trace
- Exit codes (0 = task complete, 2 = task not complete)

Your job is to fill in the **agent loop** — the part that decides *what tools to call and when*. That lives in three files you'll edit:

| file | what's in it | what you do |
|---|---|---|
| `my_agent/agent.py` | `CodingAgent.run(task)` — the per-task loop | Implement the loop. Decide what the model sees and how tool results flow back. |
| `my_agent/model.py` | `build_strands_agent(...)` — wires up Bedrock | Implement using `strands.models.bedrock.BedrockModel`. |
| `my_agent/tools.py` | `build_toolset(...)` — read_file, edit_file, etc. | Implement tools. Every one must call `trace.tool(name, input)`. |

The skeleton is structured so the **no-op contract check** passes immediately (it short-circuits in `agent.py` for `NOOP_CONTRACT_CHECK`). That means you can run the contract validator now to confirm the eval can talk to your agent, *before* you write any real code. Then you fill in the stubs incrementally, re-running the smoke test as you go.

## What's in the skeleton

```
my_agent/
├── __init__.py        # nothing to change
├── __main__.py        # CLI plumbing — DO NOT EDIT
├── trace.py           # trace recorder — DO NOT EDIT
├── agent.py           # CodingAgent.run — YOU FILL IN
├── model.py           # build_strands_agent — YOU FILL IN
└── tools.py           # build_toolset — YOU FILL IN
```

Read `__main__.py` to understand the contract the harness already implements. Then read the TODOs in `agent.py`, `model.py`, `tools.py`.

## Two ways to build it

You can do this **with Claude Code** (read the file, ask it to fill in the TODOs) or **by hand**. Either way, the smoke test cells below check your progress at two milestones: (1) contract check, (2) easy task smoke test.

## Step 1 — Read the existing skeleton

Below: the three files you'll edit. Skim them first. Notice that the no-op contract task is already handled in `agent.py` — that's why the contract check passes before you write anything else.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

WORKSHOP_DIR = Path.cwd().resolve()
AGENT_DIR = WORKSHOP_DIR / 'my_agent'

for fname in ('agent.py', 'model.py', 'tools.py'):
    src = (AGENT_DIR / fname).read_text()
    display(Markdown(f'### `my_agent/{fname}`\n\n```python\n{src}\n```'))

## Step 2 — Run the contract check (passes before you write any code)

The validator runs the harness against a "no-op" task that says "do nothing, just exit". Because `CodingAgent.run` short-circuits for `NOOP_CONTRACT_CHECK`, this passes immediately — confirming the eval can spawn the agent, get a diff out at `--out`, get a trace out at `--trace-out`, and read both back.

This is the most boring cell in the whole notebook. It's also the one that catches "I broke the harness" regressions instantly when you start editing.

In [ ]:
import sys
import yaml
sys.path.insert(0, '.')
from utils.workspace import create_workspace
from validators.agent import validate_agent_contract

tasks_doc = yaml.safe_load((WORKSHOP_DIR / 'scaffolding' / 'tasks' / 'tasks.yaml').read_text())
repo_meta = tasks_doc['repo']

ws = create_workspace(
    repo_url=repo_meta['url'],
    pinned_sha=repo_meta['pinned_sha'],
    agent='contract_check',
    task_id='NOOP',
)
v = validate_agent_contract(
    module='my_agent',
    repo_path=ws.repo_path,
    cwd=WORKSHOP_DIR,
    timeout=60,
)
print(v.report())
ws.cleanup()
assert v.passed, 'Contract check failed. The skeleton should pass before you edit anything — see errors above.'

## Step 3 — Fill in the stubs

Time to write code. Open the three files in your editor (or in Claude Code) and do the following, in this order:

### 3a. `my_agent/tools.py` — start with the basics

Implement at minimum:
- `read_file(path, start_line=1, end_line=-1)` — return file contents (or a line range)
- `edit_file(path, old_string, new_string)` — unique-substring replace; reject if `old_string` doesn't match exactly once
- `run_grep(pattern, path='.')` — `subprocess.run(['grep', '-rn', pattern, path], cwd=repo_path)`

Every tool body's **first line** is `trace.tool(name, input_dict)`. Without that, the eval cannot score tool-call quality (notebook 07's tool_call_score column will be 0 for everything).

You can stop here — read/edit/grep is enough to solve T01, T02, and T03. Add `find_callers` / `find_dependencies` (MCP-backed) only if you want to score well on T08 (nav-only) and harder tasks.

### 3b. `my_agent/model.py` — wire up Bedrock

```python
from strands import Agent
from strands.models.bedrock import BedrockModel
from .tools import build_toolset

def build_strands_agent(repo_path, trace, model_id=DEFAULT_MODEL_ID):
    model = BedrockModel(model_id=model_id, temperature=0)
    tools = build_toolset(repo_path=repo_path, trace=trace)
    return Agent(model=model, tools=tools)
```

Strands' `Agent` already implements the LLM ↔ tool loop. You don't write that yourself.

### 3c. `my_agent/agent.py` — wire it together

In `CodingAgent.__init__`, call `build_strands_agent(repo_path, trace)` and store the result. In `CodingAgent.run`, build a prompt from the task, invoke the agent, and translate the response into a `RunResult`.

The minimum viable `run` is about 10 lines:
```python
def run(self, task):
    if task.get('id') == 'NOOP_CONTRACT_CHECK':
        return RunResult(completed=True, summary='noop')
    prompt = self._build_prompt(task)
    response = self.agent(prompt)
    # Optional: extract token usage from response.metrics and call self.trace.usage(...)
    completed = 'TASK_COMPLETE' in str(response)
    return RunResult(completed=completed, summary=str(response)[:200])
```

When you've made changes, jump down to **Step 4** and run the smoke test.

## Step 4 — Smoke test on T01 (the easiest task)

T01 is a single-line deletion of a stray `print()`. If your agent can solve T01, the wiring is right and you're ready for the full eval.

This is the first cell that incurs Bedrock spend (a few cents). Run it after you've finished Step 3.

In [ ]:
from utils.runners import run_user_agent

t01 = next(t for t in tasks_doc['tasks'] if t['id'] == 'T01_remove_stray_print_chat_workflow')

ws = create_workspace(
    repo_url=repo_meta['url'],
    pinned_sha=repo_meta['pinned_sha'],
    agent='my_agent',
    task_id=t01['id'],
)
out = run_user_agent(
    task=t01,
    workspace=ws,
    module='my_agent',
    tasks_file=(WORKSHOP_DIR / 'scaffolding' / 'tasks' / 'tasks.yaml').resolve(),
    cwd=WORKSHOP_DIR,
    timeout=300,
)
print(f'elapsed={out.elapsed_s:.1f}s  exit={out.exit_code}  diff_chars={len(out.diff)}  tool_calls={len(out.tool_trace)}')
if out.error:
    print('ERROR:', out.error)
print()
print('--- Diff ---')
print(out.diff)
print()
print('--- Tool trace (first 10) ---')
for entry in out.tool_trace[:10]:
    print(entry)
ws.cleanup()

## Step 5 — Iterate

You'll see one of three outcomes. The fix for each:

| symptom | likely cause | fix |
|---|---|---|
| Empty diff, exit=2, no tool calls | Agent answered without invoking tools | System prompt isn't pushing the model to use tools. Add an explicit "you MUST read the file before editing" instruction in `_build_prompt`. |
| Empty diff, exit=2, several tool calls | Tools are read-only or `edit_file` is silently failing | Check `edit_file` returns a useful error string when `old_string` doesn't match uniquely; the model needs that signal to retry. |
| Non-empty diff but the print is still there | Agent hit a different file, or made an unrelated change | The prompt isn't scoped enough. Pass `task['affected_paths']` and `task['relevant_files']` into the prompt. |

When T01 looks right (single-line deletion of `print(response)`, exit=0, ~3-5 tool calls), you're done with Step 5. Move on to evaluation.

## What to skip on the first pass

- **MCP tools (`find_callers`, `find_dependencies`)** — useful but optional. The eval doesn't require them; tasks that need them will just score lower. Add later.
- **Token usage capture** — purely informational. The autonomous eval uses wall-clock seconds for cross-agent comparison; tokens are a within-agent tuning signal.
- **Q&A mode for the pair-programmer eval** — notebook 06 includes the Q&A invocation but you can skip it and evaluate only the autonomous axis.

## Next

Once your agent solves T01, you have everything needed for the full eval. Move on to:

- **`06 pair-programmer eval.ipynb`** — Q&A and IR scoring (optional for `my_agent` if you didn't implement Q&A mode)
- **`07 autonomous eval and report.ipynb`** — the headline scorecard across all 9 tasks for all 3 agents